

# Project : LangGraph Agent System

A multi-step business analyst agent built with LangGraph: planner, tool-using agent, reviewer with a retry loop, and conversation memory across turns. Uses Groq as the LLM.

## What this notebook does
- Builds 4 tools the agent can call: sales lookup (by region, month and product), top products, inventory check, calculator
- Builds a LangGraph state machine with nodes: planner, agent, tools, reviewer
- Uses conditional edges (tool call or finish, approve or retry) and a loop limit
- Adds MemorySaver so follow-up questions remember earlier turns
- Prints the graph, streams every step, and shows which tools were called
- Runs test tasks and checks both the tools used and the final numbers against the data

## How to run
1. Upload this file to Google Colab (or open it from Drive)
2. Runtime → Run all, or run each cell one by one
3. If a step fails, run the cell again, then restart the runtime once and run again
4. This notebook makes about 30 API calls.

## Step 1: Install libraries

In [1]:
!pip install -q langgraph langchain-openai langchain-core pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 9.8 MB/s eta 0:00:00


## Step 2: Connect to Groq (the AI part)
Paste your Groq API key when it asks. The key is hidden while you type.

In [2]:
import time
from getpass import getpass
from openai import OpenAI

GROQ_KEY = getpass("Paste your Groq API key: ")
client = OpenAI(api_key=GROQ_KEY, base_url="https://api.groq.com/openai/v1")
MODEL = "openai/gpt-oss-120b"

def ask_llm(question, system="You are a helpful analyst.", model=MODEL):
    for tries in range(3):
        try:
            r = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": question}],
                max_tokens=1500,
                temperature=0.3,
            )
            return r.choices[0].message.content
        except Exception as e:
            print("llm error, trying again:", str(e)[:100])
            time.sleep(5)
    return "LLM not available right now."

print(ask_llm("Say hi in five words."))

Paste your Groq API key: ··········
Hey there, great to see!


## Step 3: Business data and tools
The agent can only answer using these tools. Sales data is made up.

In [3]:
import json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(16)

regions = ["North", "South", "East", "West"]
products = {"Laptop": 55000, "Phone": 25000, "Tablet": 30000, "Headphones": 3000, "Monitor": 12000}
rows = []
for m in ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]:
    for r in regions:
        for p, price in products.items():
            units = int(np.random.randint(10, 80))
            rows.append({"month": m, "region": r, "product": p, "units": units, "revenue": units * price})
sales = pd.DataFrame(rows)
inventory = {"Laptop": 120, "Phone": 340, "Tablet": 45, "Headphones": 900, "Monitor": 18}
print(sales.shape)
sales.head()

(120, 5)


,month,region,product,units,revenue
0,Jan,North,Laptop,51,2805000
1,Jan,North,Phone,79,1975000
2,Jan,North,Tablet,75,2250000
3,Jan,North,Headphones,78,234000
4,Jan,North,Monitor,41,492000


In [4]:
from langchain_core.tools import tool

@tool
def get_sales(region: str, month: str = "all", product: str = "all") -> str:
    """Get total units and revenue (Indian rupees) for sales. region is North, South, East, West or 'all'. month is Jan to Jun (full names like March also work) or 'all'. product is Laptop, Phone, Tablet, Headphones, Monitor or 'all'."""
    df = sales
    if region.lower() != "all":
        df = df[df["region"].str.lower() == region.lower()]
    if month.lower() != "all":
        df = df[df["month"].str.lower() == month.lower()[:3]]
    if product.lower() != "all":
        df = df[df["product"].str.lower() == product.lower()]
    if len(df) == 0:
        return "No data found for that region, month or product."
    return json.dumps({"region": region, "month": month, "product": product, "units": int(df["units"].sum()), "revenue_inr": int(df["revenue"].sum())})

@tool
def top_products(n: int = 3) -> str:
    """Get the top n products by total revenue across all regions and months."""
    top = sales.groupby("product")["revenue"].sum().sort_values(ascending=False).head(int(n))
    return json.dumps({k: int(v) for k, v in top.items()})

@tool
def check_inventory(product: str) -> str:
    """Get units in stock. product is Laptop, Phone, Tablet, Headphones or Monitor. You can pass several products at once separated by commas, like 'Monitor, Tablet'."""
    results = {}
    for name in str(product).split(","):
        name = name.strip()
        for key, qty in inventory.items():
            if key.lower() == name.lower():
                results[key] = qty
    if len(results) == 0:
        return "Product not found."
    return json.dumps(results)

@tool
def calculate(expression: str) -> str:
    """Do maths. Input is an expression like '(120000 - 90000) / 90000 * 100'. Only numbers and + - * / ( ) . are allowed."""
    allowed = "0123456789+-*/(). "
    if any(ch not in allowed for ch in expression):
        return "Only numbers and + - * / ( ) are allowed."
    try:
        return str(round(eval(expression), 4))
    except Exception as e:
        return "Could not calculate: " + str(e)

tools = [get_sales, top_products, check_inventory, calculate]
print(get_sales.invoke({"region": "North", "month": "Jan"}))
print(get_sales.invoke({"region": "North", "month": "March"}))
print(get_sales.invoke({"region": "West", "month": "all", "product": "Tablet"}))
print(top_products.invoke({"n": 2}))
print(check_inventory.invoke({"product": "Tablet"}))
print(check_inventory.invoke({"product": "Monitor, Tablet"}))
print(calculate.invoke({"expression": "(120000 - 90000) / 90000 * 100"}))

{"region": "North", "month": "Jan", "product": "all", "units": 324, "revenue_inr": 7756000}
{"region": "North", "month": "March", "product": "all", "units": 251, "revenue_inr": 6711000}
{"region": "West", "month": "all", "product": "Tablet", "units": 278, "revenue_inr": 8340000}
{"Laptop": 55550000, "Tablet": 37380000}
{"Tablet": 45}
{"Monitor": 18, "Tablet": 45}
33.3333


## Step 4: Build the graph
Flow: planner → agent ⇄ tools → reviewer → (retry once, or finish)

In [5]:
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model=MODEL, api_key=GROQ_KEY, base_url="https://api.groq.com/openai/v1", temperature=0)
llm_with_tools = llm.bind_tools(tools)

class State(MessagesState):
    plan: str
    review: str
    retries: int

def planner(state: State):
    question = state["messages"][-1].content
    prompt = ("You plan work for a business analyst agent. Tools: get_sales(region, month, product), top_products(n), check_inventory(product), calculate(expression). "
              "Write a plan of at most 4 short numbered steps. Do only what the request asks, with no extra analysis. Request: " + str(question))
    plan = llm.invoke([HumanMessage(content=prompt)]).content
    return {"plan": plan, "retries": 0}

def agent(state: State):
    system = ("You are a business analyst agent. Use the tools to get facts and never guess numbers. All money is in Indian rupees, so write it with the rupee sign. Do only what the question asks. If a tool cannot give what you need, say so instead of using a wrong number. If you need several independent facts, ask for all the tool calls in one step. Never repeat a tool call with the same arguments. "
              "Follow this plan when useful:\n" + state.get("plan", "") +
              "\nWhen you have everything, give a short final answer with the numbers.")
    reply = llm_with_tools.invoke([SystemMessage(content=system)] + state["messages"])
    return {"messages": [reply]}

def reviewer(state: State):
    msgs = state["messages"]
    last = msgs[-1].content
    q_index = 0
    for i, m in enumerate(msgs):
        if isinstance(m, HumanMessage) and not str(m.content).startswith("Reviewer feedback:"):
            q_index = i
    question = msgs[q_index].content
    tool_results = [str(m.content) for m in msgs[q_index:] if isinstance(m, ToolMessage)]
    prompt = ("You review an analyst answer. Check 3 things: (1) every part of the question is answered, "
              "(2) every number in the answer comes from the tool results or from a correct calculation of them, "
              "(3) the calculation matches what was asked (right product, region, period and total). "
              "Reply APPROVE unless you can point to a specific number or claim in the answer that contradicts the tool results or the question. Only then reply REVISE: <the exact contradiction>. Do not use your own memory of the data, only the tool results.\n\nQuestion: " + str(question) +
              "\n\nTool results:\n" + "\n".join(tool_results) + "\n\nAnswer: " + str(last))
    verdict = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return {"review": verdict, "retries": state.get("retries", 0) + 1}

def after_review(state: State):
    if state["review"].upper().startswith("REVISE") and state["retries"] < 2:
        return "revise"
    return "done"

def add_feedback(state: State):
    return {"messages": [HumanMessage(content="Reviewer feedback: " + state["review"] + ". Fix the problem using the facts you already have. Call a tool again only if a fact is really missing, and never repeat a call with the same arguments.")]}

builder = StateGraph(State)
builder.add_node("planner", planner)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_node("reviewer", reviewer)
builder.add_node("feedback", add_feedback)

builder.add_edge(START, "planner")
builder.add_edge("planner", "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": "reviewer"})
builder.add_edge("tools", "agent")
builder.add_conditional_edges("reviewer", after_review, {"revise": "feedback", "done": END})
builder.add_edge("feedback", "agent")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

try:
    print(graph.get_graph().draw_mermaid())
except Exception as e:
    print("Could not draw graph:", str(e)[:80])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	agent(agent)
	tools(tools)
	reviewer(reviewer)
	feedback(feedback)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	agent -. &nbsp;__end__&nbsp; .-> reviewer;
	agent -.-> tools;
	feedback --> agent;
	planner --> agent;
	reviewer -. &nbsp;done&nbsp; .-> __end__;
	reviewer -. &nbsp;revise&nbsp; .-> feedback;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 5: Run the agent and watch every step

In [6]:
from langgraph.errors import GraphRecursionError

def run_agent(question, thread_id):
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 60}
    tool_calls = []
    final_answer = ""
    print("QUESTION:", question)
    try:
        for update in graph.stream({"messages": [HumanMessage(content=question)]}, config, stream_mode="updates"):
            for node, data in update.items():
                if node == "planner":
                    print("  [planner] plan:", data["plan"].replace("\n", " | ")[:200])
                elif node == "agent":
                    msg = data["messages"][-1]
                    if getattr(msg, "tool_calls", None):
                        for tc in msg.tool_calls:
                            tool_calls.append(tc["name"])
                            print("  [agent] calls tool:", tc["name"], tc["args"])
                    else:
                        final_answer = msg.content
                        print("  [agent] answer drafted")
                elif node == "tools":
                    for m in data["messages"]:
                        print("  [tool result]", str(m.content)[:120])
                elif node == "reviewer":
                    print("  [reviewer]", data["review"][:100])
    except GraphRecursionError:
        print("  [stopped] the agent used too many steps (loop limit). Showing the last answer it drafted.")
    print("FINAL ANSWER:", final_answer)
    print("Tools used:", tool_calls)
    print()
    return final_answer, tool_calls

a1, t1 = run_agent("What was the total revenue in the North region in March?", "chat1")
a2, t2 = run_agent("Which are the top 3 products by revenue, and what percent of total revenue across all 5 products is the number 1 product? Use the calculator.", "chat2")

QUESTION: What was the total revenue in the North region in March?
  [planner] plan: 1. Use **top_products** to obtain a list of all products.   | 2. For each product, call **get_sales("North", "March", product)** to retrieve its revenue.   | 3. Collect all revenue values and feed the
  [agent] calls tool: top_products {'n': 5}
  [tool result] {"Laptop": 55550000, "Tablet": 37380000, "Phone": 29950000, "Monitor": 12432000, "Headphones": 3564000}
  [agent] calls tool: get_sales {'month': 'March', 'product': 'Laptop', 'region': 'North'}
  [tool result] {"region": "North", "month": "March", "product": "Laptop", "units": 51, "revenue_inr": 2805000}
  [agent] calls tool: get_sales {'month': 'March', 'product': 'Tablet', 'region': 'North'}
  [tool result] {"region": "North", "month": "March", "product": "Tablet", "units": 56, "revenue_inr": 1680000}
  [agent] calls tool: get_sales {'month': 'March', 'product': 'Phone', 'region': 'North'}
  [tool result] {"region": "North", "month": "March", 

## Step 6: Memory test
Same thread id means the agent remembers the earlier question. A different thread id starts fresh.

In [7]:
a3, t3 = run_agent("And what about the South region in the same month?", "chat1")
a4, t4 = run_agent("And what about the South region in the same month?", "brand_new_chat")

QUESTION: And what about the South region in the same month?
  [planner] plan: 1. Identify the month referenced in the previous query.   | 2. Call `get_sales("South", <month>, "all")` to retrieve sales data for the South region in that month.   | 3. Summarize the total sales and
  [agent] calls tool: get_sales {'month': 'March', 'region': 'South'}
  [tool result] {"region": "South", "month": "March", "product": "all", "units": 171, "revenue_inr": 3004000}
  [agent] answer drafted
  [reviewer] APPROVE
FINAL ANSWER: The total revenue for the South region in March is **₹3,004,000**.
Tools used: ['get_sales']

QUESTION: And what about the South region in the same month?
  [planner] plan: 1. Determine the month referenced in the previous query.   | 2. Call `get_sales("South", <month>, "all")` to retrieve sales data for the South region for that month.   | 3. Summarize the total sales a
  [agent] answer drafted
  [reviewer] REVISE: The answer does not provide the South region data for the re

## Step 7: Multi-tool task

In [8]:
a5, t5 = run_agent("Check inventory for Monitor and Tablet. Which one is lower in stock? Then, using the average monthly Tablet units sold in the West region over all 6 months, how many months of Tablet sales does the current Tablet stock cover?", "chat3")

QUESTION: Check inventory for Monitor and Tablet. Which one is lower in stock? Then, using the average monthly Tablet units sold in the West region over all 6 months, how many months of Tablet sales does the current Tablet stock cover?
  [planner] plan: 1. Use **check_inventory** for “Monitor” and “Tablet” to obtain current stock levels.   | 2. Compare the two stock values and identify which product has the lower inventory.   | 3. Retrieve Tablet sal
  [agent] calls tool: check_inventory {'product': 'Monitor, Tablet'}
  [tool result] {"Monitor": 18, "Tablet": 45}
  [agent] calls tool: get_sales {'month': 'January', 'product': 'Tablet', 'region': 'West'}
  [tool result] {"region": "West", "month": "January", "product": "Tablet", "units": 38, "revenue_inr": 1140000}
  [agent] calls tool: get_sales {'month': 'all', 'product': 'Tablet', 'region': 'West'}
  [tool result] {"region": "West", "month": "all", "product": "Tablet", "units": 278, "revenue_inr": 8340000}
  [agent] calls tool: calcu

## Step 8: Checks
We check the agent used the right tools AND that the final numbers match the data. A pretty answer with a wrong number must fail.

In [9]:
import re

def numbers_in(text):
    cleaned = str(text).replace(",", "")
    return [float(x) for x in re.findall(r"\d+\.?\d*", cleaned)]

def has_number(text, target, tol=0.02):
    for n in numbers_in(text):
        if abs(n - target) <= abs(target) * tol + 0.0001:
            return True
    return False

expected_north_march = int(sales[(sales["region"] == "North") & (sales["month"] == "Mar")]["revenue"].sum())
total_all = int(sales["revenue"].sum())
laptop_total = int(sales[sales["product"] == "Laptop"]["revenue"].sum())
expected_share = laptop_total / total_all * 100
west_tablet_units = int(sales[(sales["region"] == "West") & (sales["product"] == "Tablet")]["units"].sum())
expected_cover = inventory["Tablet"] / (west_tablet_units / 6)

print("Expected North March revenue:", expected_north_march)
print("Expected Laptop share of all revenue (%):", round(expected_share, 2))
print("Expected months of Tablet cover (West):", round(expected_cover, 2))

checks = [
    {"name": "sales lookup used get_sales", "ok": "get_sales" in t1},
    {"name": "North March revenue number is correct", "ok": has_number(a1, expected_north_march, 0.001)},
    {"name": "top products used top_products", "ok": "top_products" in t2},
    {"name": "calculator used for percentage", "ok": "calculate" in t2},
    {"name": "Laptop share of ALL revenue is correct", "ok": has_number(a2, expected_share, 0.02)},
    {"name": "inventory task used check_inventory", "ok": "check_inventory" in t5},
    {"name": "Monitor named as lower stock", "ok": "monitor" in a5.lower()},
    {"name": "Tablet months of cover is correct", "ok": has_number(a5, expected_cover, 0.05)},
]

check_df = pd.DataFrame(checks)
print()
print(check_df.to_string(index=False))
print("\nPassed", int(check_df["ok"].sum()), "of", len(check_df))

Expected North March revenue: 6711000
Expected Laptop share of all revenue (%): 40.0
Expected months of Tablet cover (West): 0.97

                                  name   ok
           sales lookup used get_sales True
 North March revenue number is correct True
        top products used top_products True
        calculator used for percentage True
Laptop share of ALL revenue is correct True
   inventory task used check_inventory True
          Monitor named as lower stock True
     Tablet months of cover is correct True

Passed 8 of 8


## Step 9: Key insights and export

In [10]:
print("KEY INSIGHTS")
print("1. The graph has 5 nodes: planner, agent, tools, reviewer, feedback, with 2 conditional edges (tool call or finish, approve or retry)")
print("2. Tools used per task:", {"task1": t1, "task2": t2, "task5": t5})
print("3. Memory test: read the two answers to the follow-up question. The same-thread answer should use March, the new-thread answer has no idea which month you mean.")
print("   same thread:", a3[:150])
print("   new thread :", a4[:150])
print("4. Checks passed:", int(check_df["ok"].sum()), "of", len(check_df))
print("5. The loop is capped (recursion_limit and reviewer retries) so the agent cannot run forever")

log = pd.DataFrame([
    {"task": "north_march", "tools": ",".join(t1), "answer": a1},
    {"task": "top3_products", "tools": ",".join(t2), "answer": a2},
    {"task": "memory_same_thread", "tools": ",".join(t3), "answer": a3},
    {"task": "memory_new_thread", "tools": ",".join(t4), "answer": a4},
    {"task": "inventory_cover", "tools": ",".join(t5), "answer": a5},
])
log.to_csv("langgraph_agent_runs.csv", index=False)
print("\nSaved langgraph_agent_runs.csv")

KEY INSIGHTS
1. The graph has 5 nodes: planner, agent, tools, reviewer, feedback, with 2 conditional edges (tool call or finish, approve or retry)
2. Tools used per task: {'task1': ['top_products', 'get_sales', 'get_sales', 'get_sales', 'get_sales', 'get_sales', 'calculate'], 'task2': ['top_products', 'calculate'], 'task5': ['check_inventory', 'get_sales', 'get_sales', 'calculate', 'calculate']}
3. Memory test: read the two answers to the follow-up question. The same-thread answer should use March, the new-thread answer has no idea which month you mean.
   same thread: The total revenue for the South region in March is **₹3,004,000**.
   new thread : I’m still missing the month you’d like the South‑region sales data for. Could you let me know which month (e.g., January, February, March, April, May,
4. Checks passed: 8 of 8
5. The loop is capped (recursion_limit and reviewer retries) so the agent cannot run forever

Saved langgraph_agent_runs.csv



---
Built by **Akshat Kesharwani** | [GitHub](https://github.com/akshatkesharwani-info) | [Portfolio](https://akshatkesharwani-info.github.io/)